# 第三课：重新走进 Transformer，理解 Attention

28页PPT、60分钟。实验01–03复习前两课；实验04–08展开主算例与三个干预；
实验09把RoPE接回计算。主例位置从0开始，第三个位置为2。

In [1]:
from pathlib import Path
import sys
import math
import torch

ROOT = Path.cwd()
if (ROOT / "class3" / "code").is_dir():
    ROOT = ROOT / "class3"
sys.path.insert(0, str(ROOT / "code"))
from attention import X, W_Q, W_K, W_V, project, attention, apply_rope
from review import build_review

torch.set_printoptions(precision=4, sci_mode=False)
review = build_review()
x = X.clone()
w_q, w_k, w_v = W_Q.clone(), W_K.clone(), W_V.clone()

## 实验01 · BPE与Embedding · PPT 3–5

每轮统计当前的相邻片段；编码执行已冻结的规则。对照最终两个token与中间五个token。

真实ByteLevel配置另行加载，查看“我喜欢喝牛奶。”的ID。梯度小表使用独立的算术ID，
人为指定上游梯度，以观察同一参数行的累加；第二课bigram记录展示预测任务的真实训练结果。

In [2]:
for row in review["bpe"]["training"]:
    print("训练rank", row["rank"], row["pair"], "频次", row["count"])
for row in review["bpe"]["encode_trace"]:
    print("编码rank", row["rank"], " / ".join(row["tokens"]))
print("冻结的真实配置：", review["real_tokenizer"])
e = review["embedding"]
print("算术小表IDs：", e["ids"], "查出：", e["vectors"])
print("参数行梯度：", e["gradient"])
print("SGD前：", e["weight"], "SGD后：", e["after"])
print("第二课bigram loss：", e["bigram_loss_start"], "→", e["bigram_loss_end"])

训练rank 0 ('喜', '欢') 频次 10
训练rank 1 ('喜欢', '喝') 频次 10
训练rank 2 ('喜欢喝', '奶') 频次 8
训练rank 3 ('喜欢喝奶', '茶') 频次 8
编码rank None 你 / 喜 / 欢 / 喝 / 奶 / 茶
编码rank 0 你 / 喜欢 / 喝 / 奶 / 茶
编码rank 1 你 / 喜欢喝 / 奶 / 茶
编码rank 2 你 / 喜欢喝奶 / 茶
编码rank 3 你 / 喜欢喝奶茶
冻结的真实配置： {'source': 'class2/assets/tokenizer_320.json', 'text': '我喜欢喝牛奶。', 'tokens': ['æĪĳåĸľæ¬¢åĸĿ', 'çīĽå¥¶', 'ãĢĤ'], 'ids': [300, 304, 262], 'decoded': '我喜欢喝牛奶。'}
算术小表IDs： [1, 1, 2] 查出： [[0.2, -0.1], [0.2, -0.1], [0.5, 0.3]]
参数行梯度： [[0.0, 0.0], [1.0, 2.0], [3.0, 4.0]]
SGD前： [[0.0, 0.0], [0.2, -0.1], [0.5, 0.3]] SGD后： [[0.0, 0.0], [0.1, -0.30000000000000004], [0.19999999999999998, -0.10000000000000003]]
第二课bigram loss： 5.7564873695373535 → 0.355943888425827


## 实验02 · 两支箭头一起转 · PPT 6

图解初始q=k=[1,0]，每位置转30°。位置1/3移到3/5，角度差仍60°，点积都为0.5。
相加式正弦PE则根据位置生成同维向量，直接加到内容向量上。

In [3]:
p = review["rope_intuition"]
print("原位置：", p["positions"], "方向：", p["angles_degrees"], "点积：", p["dot"])
print("移动后：", p["shifted_positions"], "方向：", p["shifted_angles_degrees"],
      "点积：", p["shifted_dot"])
print("四维相加式正弦PE：", p["pe_width4"])

原位置： [1, 3] 方向： [30, 90] 点积： 0.5
移动后： [3, 5] 方向： [90, 150] 点积： 0.4999999999999999
四维相加式正弦PE： [[0.0, 1.0, 0.0, 1.0], [0.8414709848078965, 0.5403023058681398, 0.009999833334166664, 0.9999500004166653], [0.9092974268256817, -0.4161468365471424, 0.01999866669333308, 0.9998000066665778]]


## 实验03 · 逐项加权汇总 · PPT 8–9、22

先看三个位置各自贡献什么，再沿第0维求和，得到喝这个位置的新向量。
最后保留两候选概率，PPT第22页对比读取比例与预测概率。
这些旧例使用第一课相同的人工参数，后半节才切换到新的Q/K/V投影。

In [4]:
old = review["first_lesson"]
values = torch.tensor(old["values"], dtype=torch.float64)
weights_review = torch.tensor(old["weights"], dtype=torch.float64)
weighted = weights_review[:, None] * values
context = weighted.sum(dim=0)
print("每个位置的贡献：\n", weighted)
print("按列求和，得到喝的新向量：", context)
for name in ("scores", "weights", "values", "context"):
    print(name, "=", old[name])
print("候选：", old["candidate_labels"])
print("候选分数：", old["lm_logits"])
print("候选概率：", old["lm_probabilities"])

每个位置的贡献：
 tensor([[0.2483, 0.0000],
        [0.0000, 0.2483],
        [0.5035, 0.5035]], dtype=torch.float64)
按列求和，得到喝的新向量： tensor([0.7517, 0.7517], dtype=torch.float64)
scores = [0.7071067811865475, 0.7071067811865475, 1.414213562373095]
weights = [0.24825507825772308, 0.24825507825772308, 0.5034898434845538]
values = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
context = [0.7517449217422769, 0.7517449217422769]
候选： ['酒', '牛奶']
候选分数： [1.5034898434845538, 0.7517449217422769]
候选概率： [0.6795587899546517, 0.3204412100453483]


## 实验04 · 三个可训练投影 · PPT 13–15

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.
$$

本课采用行向量右乘；三张2×2矩阵是便于手算的参数取值。实际训练会更新这些矩阵。
主算例暂不叠加位置，便于逐项核对；实验09接回真实RoPE。

In [5]:
q = x @ w_q
k = x @ w_k
v = x @ w_v
for name, tensor in {"X": X, "W_Q": W_Q, "W_K": W_K, "W_V": W_V,
                     "Q": q, "K": k, "V": v}.items():
    print(name, "\n", tensor)
print("喝的V：", X[-1] @ W_V)

X 
 tensor([[1., 0.],
        [0., 1.],
        [1., 1.]], dtype=torch.float64)
W_Q 
 tensor([[1., 0.],
        [0., 2.]], dtype=torch.float64)
W_K 
 tensor([[1., 1.],
        [0., 1.]], dtype=torch.float64)
W_V 
 tensor([[ 1., -1.],
        [ 2.,  0.]], dtype=torch.float64)
Q 
 tensor([[1., 0.],
        [0., 2.],
        [1., 2.]], dtype=torch.float64)
K 
 tensor([[1., 1.],
        [0., 1.],
        [1., 2.]], dtype=torch.float64)
V 
 tensor([[ 1., -1.],
        [ 2.,  0.],
        [ 3., -1.]], dtype=torch.float64)
喝的V： tensor([ 3., -1.], dtype=torch.float64)


## 实验05 · 一行到完整矩阵 · PPT 16–21、26

行是query，列是key。第三行点积为[3,2,5]；每行按自己的可读范围归一化。

$$
S=QK^\mathsf{T}/\sqrt{d_k},\quad
A=\operatorname{softmax}_{j}(S+M),\quad Z=AV.
$$

In [6]:
raw_scores = q @ k.T
scores = raw_scores / math.sqrt(q.shape[-1])
future = torch.ones(3, 3, dtype=torch.bool).triu(1)
masked = scores.masked_fill(future, -torch.inf)
weights = masked.softmax(dim=-1)
output = weights @ v

for name, tensor in {"原始分数": raw_scores, "缩放分数": scores,
                     "mask后": masked, "权重": weights, "输出": output}.items():
    print(name, "\n", tensor)
print("每行权重和：", weights.sum(-1))
print("第二行手算：", .5*v[0] + .5*v[1])
print("第三行不缩放/缩放后的权重：", raw_scores[-1].softmax(-1), weights[-1])

原始分数 
 tensor([[1., 0., 1.],
        [2., 2., 4.],
        [3., 2., 5.]], dtype=torch.float64)
缩放分数 
 tensor([[0.7071, 0.0000, 0.7071],
        [1.4142, 1.4142, 2.8284],
        [2.1213, 1.4142, 3.5355]], dtype=torch.float64)
mask后 
 tensor([[0.7071,   -inf,   -inf],
        [1.4142, 1.4142,   -inf],
        [2.1213, 1.4142, 3.5355]], dtype=torch.float64)
权重 
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.1784, 0.0879, 0.7337]], dtype=torch.float64)
输出 
 tensor([[ 1.0000, -1.0000],
        [ 1.5000, -0.5000],
        [ 2.5553, -0.9121]], dtype=torch.float64)
每行权重和： tensor([1., 1., 1.], dtype=torch.float64)
第二行手算： tensor([ 1.5000, -0.5000], dtype=torch.float64)
第三行不缩放/缩放后的权重： tensor([0.1142, 0.0420, 0.8438], dtype=torch.float64) tensor([0.1784, 0.0879, 0.7337], dtype=torch.float64)


## 实验06 · 只改V · PPT 23

Q/K保持，修改已经生成的第0行V。先预测权重会不会变，再核对解析关系：

$$\Delta z_i=A_{i,0}\Delta v_0.$$

In [7]:
changed_v = v.clone()
delta_v = v.new_tensor([2., 0.])
changed_v[0] += delta_v
result_v = attention(q, k, changed_v)
print("权重保持：", torch.allclose(weights, result_v["weights"]))
print("实际输出变化：\n", result_v["output"] - output)
print("解析预测：\n", weights[:, 0:1] * delta_v)
torch.testing.assert_close(result_v["output"] - output, weights[:, 0:1] * delta_v)

权重保持： True
实际输出变化：
 tensor([[2.0000, 0.0000],
        [1.0000, 0.0000],
        [0.3567, 0.0000]], dtype=torch.float64)
解析预测：
 tensor([[2.0000, 0.0000],
        [1.0000, 0.0000],
        [0.3567, 0.0000]], dtype=torch.float64)


## 实验07 · 只改K · PPT 24

Q/V保持，K[0]加[1,0]。主要观察第三行：它对第0列的原始分数从3变4。
第一行只有一个可读位置，第二行Q=[0,2]与本次改变量正交，因此这两行权重保持。

In [8]:
changed_k = k.clone()
changed_k[0] += k.new_tensor([1., 0.])
result_k = attention(q, changed_k, v)
print("原权重：\n", weights)
print("新权重：\n", result_k["weights"])
print("新输出：\n", result_k["output"])
print("第三行读取第0列：", weights[2, 0].item(), "→",
      result_k["weights"][2, 0].item())

原权重：
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.1784, 0.0879, 0.7337]], dtype=torch.float64)
新权重：
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3057, 0.0743, 0.6200]], dtype=torch.float64)
新输出：
 tensor([[ 1.0000, -1.0000],
        [ 1.5000, -0.5000],
        [ 2.3143, -0.9257]], dtype=torch.float64)
第三行读取第0列： 0.17837015472760462 → 0.3056952508389745


## 实验08 · 修改未来输入 · PPT 25

位置与参数固定，把最后输入从[1,1]改为[4,-1]，重新生成Q/K/V。
比较较早两行输出；同时检查最后一行确实改变，并运行无mask对照。

In [9]:
changed_x = x.clone()
changed_x[-1] += X.new_tensor([3., -2.])
q2, k2, v2 = project(changed_x)
masked_after = attention(q2, k2, v2)["output"]
plain_before = attention(q, k, v, causal=False)["output"]
plain_after = attention(q2, k2, v2, causal=False)["output"]
print("mask开，前两行最大变化：", (masked_after[:-1] - output[:-1]).abs().max().item())
print("mask关，前两行最大变化：",
      (plain_after[:-1] - plain_before[:-1]).abs().max().item())
print("最后位置：", output[-1], "→", masked_after[-1])
torch.testing.assert_close(output[:-1], masked_after[:-1])
assert not torch.allclose(output[-1], masked_after[-1])

mask开，前两行最大变化： 0.0
mask关，前两行最大变化： 2.7935773359599616
最后位置： tensor([ 2.5553, -0.9121], dtype=torch.float64) → tensor([ 1.9965, -3.9887], dtype=torch.float64)


## 实验09 · 接回第二课的真实RoPE接口 · PPT 6、25

apply_rope复用第二课实现。按固定位置旋转Q/K，V保留，再进行因果Attention。
固定未旋转内容，整体平移位置后点积保持；修改未来输入时，较早输出仍保持。

In [10]:
positions = torch.arange(3, dtype=X.dtype)
qr, kr = apply_rope(q, positions), apply_rope(k, positions)
rope_output = attention(qr, kr, v)["output"]
shifted_q = apply_rope(q, positions + 7)
shifted_k = apply_rope(k, positions + 7)
rope_future = attention(apply_rope(q2, positions),
                        apply_rope(k2, positions), v2)["output"]
print("RoPE输出：\n", rope_output)
print("Q范数保持：", torch.allclose(q.norm(dim=-1), qr.norm(dim=-1)))
print("整体平移，点积最大差：", (qr @ kr.T - shifted_q @ shifted_k.T).abs().max().item())
print("RoPE下，未来修改不影响前两行：",
      torch.allclose(rope_output[:-1], rope_future[:-1]))
torch.testing.assert_close(rope_output[:-1], rope_future[:-1])

RoPE输出：
 tensor([[ 1.0000, -1.0000],
        [ 1.8630, -0.1370],
        [ 2.8874, -0.8987]], dtype=torch.float64)
Q范数保持： True
整体平移，点积最大差： 8.881784197001252e-16
RoPE下，未来修改不影响前两行： True


## 课堂检查与第四课

解释一行读取权重、计算一次加权V、预测三种修改的结果。
第27页多头图使用独立形状[1,3,4]，两头各2维；下一课实现多头和完整Block。
练习见 [exercises.md](exercises.md)，公式与页码对应见 [knowledge_code_map.md](knowledge_code_map.md)。